In [ ]:

import os
import requests
from dotenv import load_dotenv


In [ ]:

from langchain_community.vectorstores import Chroma

from langchain_community.document_loaders import TextLoader

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_huggingface import HuggingFaceEmbeddings

from langchain_ollama import ChatOllama


In [ ]:

load_dotenv()

SERPER_API_KEY = os.getenv("SERPER_API_KEY")

embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)


In [ ]:

DB_DIR = "chroma_db"

def create_vectorstore():
    print(" Rebuilding Chroma DB...")

    from langchain_community.document_loaders import TextLoader
    
    loader = TextLoader("usa.txt")
    documents = loader.load()

    print("\n📄 Loaded Document:")
    print(documents[0].page_content) 

    from langchain_text_splitters import RecursiveCharacterTextSplitter
    
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=300,
        chunk_overlap=50
    )

    chunks = splitter.split_documents(documents)

    print(f"\n Total Chunks Created: {len(chunks)}")

    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embedding,
        persist_directory="chroma_db"
    )

    vectorstore.persist()
    return vectorstore


vectorstore = create_vectorstore()


In [ ]:

def vector_search(query):
    docs = vectorstore.similarity_search(query, k=3)
    return "\n".join([doc.page_content for doc in docs])


In [ ]:


def serper_search(query):
    url = "https://google.serper.dev/search"

    payload = {"q": query}
    headers = {
        "X-API-KEY": SERPER_API_KEY,
        "Content-Type": "application/json"
    }

    response = requests.post(url, json=payload, headers=headers)
    data = response.json()

    results = []
    for item in data.get("organic", []):
        results.append(item.get("snippet", ""))

    return "\n".join(results[:5])


In [ ]:

def retriever_agent(query):
    
    print(" Trying Vector DB first...")
    
    docs = vectorstore.similarity_search(query, k=2)  # smaller = better
    
    if docs:
        print(" Using Vector DB")
        return "\n".join([doc.page_content for doc in docs])
    
    print(" Falling back to Web")
    return serper_search(query)


In [ ]:

llm = ChatOllama(model="llama3.2")

def response_agent(query, context):
    prompt = f"""
    Answer the question based only on the context below.

    Context:
    {context}

    Question:
    {query}

    Give a clear and concise answer.
    """
    
    response = llm.invoke(prompt)
    return response.content


In [ ]:

def agentic_rag(query):
    
    #  Retrieve context
    context = retriever_agent(query)

    #  Generate response
    answer = response_agent(query, context)

    return answer

In [ ]:

if __name__ == "__main__":
    while True:
        query = input("\n Ask a question (or type 'exit'): ")

        if query.lower() == "exit":
            break

        result = agentic_rag(query)
        print("\n Answer:\n", result)